# Module 4 - Embeddings neuroniques (PyTorch)

## Objectifs

1. Passer de numpy a **PyTorch** pour le CF.
2. Implementer **Two-Tower** (produit scalaire entre embeddings user/item).
3. Implementer **NeuralCF** (MLP sur concat des embeddings user/item, He et al. 2017).
4. Visualiser les courbes d apprentissage.
5. Comprendre les compromis : expressivite (NeuralCF) vs simplicite (Two-Tower) vs performance brute (Surprise SVD).

## Limites CPU

L entrainement est **lent sur CPU** (~20s par epoch sur 100k notes). Le notebook utilise un echantillon de 100k ratings pour rester interactif. En production, on utilise un GPU ou des batchs encore plus gros.


## 1. Setup

In [ ]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(42)
torch.manual_seed(42)

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.loaders import load_movielens_1m, temporal_split
from src.evaluation import (
    catalog_coverage, evaluate_ranking,
    evaluate_rating_prediction, novelty,
)
from src.models.neural_cf import TorchCF
from src.models.surprise_models import make_surprise_model

print(f"Torch {torch.__version__}, device: cpu")

## 2. Chargement (echantillon 100k pour vitesse)

In [ ]:
ml = load_movielens_1m(ROOT / "data" / "raw" / "ml-1m")
ratings = ml.ratings.sample(n=100000, random_state=0).reset_index(drop=True)
print(f"Sample: {len(ratings):,} ratings")

train, test = temporal_split(ratings, test_ratio=0.2)
train_sorted = train.sort_values("timestamp").reset_index(drop=True)
cut = int(len(train_sorted) * 0.9)
train_fit = train_sorted.iloc[:cut].copy()
val_fit = train_sorted.iloc[cut:].copy()

train_users = set(train.user_id.unique()); train_items = set(train.movie_id.unique())
test_eval = test[test.user_id.isin(train_users) & test.movie_id.isin(train_items)].copy()
rng = np.random.default_rng(0)
eval_users = list(rng.choice(test_eval.user_id.unique(), size=300, replace=False))
test_subset = test_eval[test_eval.user_id.isin(eval_users)]

## 3. Two-Tower : embeddings + produit scalaire

Architecture la plus simple :

$$
\\hat r_{ui} = \\mu + b_u + b_i + e_u \\cdot e_i
$$

Avec entrainement par MSE sur des mini-batchs.

In [ ]:
t0 = time.time()
twotower = TorchCF(arch="twotower", n_factors=32, n_epochs=8, batch_size=4096,
                   lr=0.005, reg=1e-5).fit(train_fit, val=val_fit)
print(f"TwoTower fit: {time.time()-t0:.1f}s")

In [ ]:
fig, ax = plt.subplots()
ax.plot(twotower.train_loss_, label="train")
ax.plot(twotower.val_rmse_, label="val")
ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE")
ax.set_title("TwoTower - courbe d apprentissage")
ax.legend()
plt.show()

## 4. NeuralCF : MLP sur les embeddings concatnes

He et al. 2017 : on concatene les embeddings user et item, puis on passe dans un MLP :

$$
\\hat r_{ui} = \\mathrm{MLP}([e_u, e_i])
$$

Le MLP peut capturer des interactions non-lineaires que le produit scalaire ne peut pas voir.

In [ ]:
t0 = time.time()
ncf = TorchCF(arch="neuralcf", n_factors=16, hidden=(64, 32), dropout=0.2,
              n_epochs=8, batch_size=4096, lr=0.001, reg=1e-4).fit(train_fit, val=val_fit)
print(f"NeuralCF fit: {time.time()-t0:.1f}s")

In [ ]:
fig, ax = plt.subplots()
ax.plot(ncf.train_loss_, label="NeuralCF train")
ax.plot(ncf.val_rmse_, label="NeuralCF val")
ax.plot(twotower.train_loss_, "--", label="TwoTower train")
ax.plot(twotower.val_rmse_, "--", label="TwoTower val")
ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE")
ax.set_title("Comparaison des courbes d apprentissage")
ax.legend()
plt.show()

## 5. Comparaison avec Surprise SVD

In [ ]:
svd = make_surprise_model("SVD", n_factors=32, n_epochs=20, random_state=0).fit(train)

pairs = test_eval[["user_id","movie_id"]].drop_duplicates().head(10_000)
pairs = pairs.merge(test_eval[["user_id","movie_id","rating"]], on=["user_id","movie_id"], how="left").dropna()

rows = []
for name, m in [("TwoTower (PyTorch)", twotower), ("NeuralCF (PyTorch)", ncf), ("Surprise SVD", svd)]:
    preds = m.predict_for_pairs(pairs[["user_id","movie_id"]])
    met = evaluate_rating_prediction(pairs, preds)
    met["model"] = name; rows.append(met)
df = pd.DataFrame(rows).set_index("model")[["rmse","mae","coverage"]]
display(df.round(4))

In [ ]:
all_recs = {}
for name, m in [("TwoTower (PyTorch)", twotower), ("NeuralCF (PyTorch)", ncf), ("Surprise SVD", svd)]:
    all_recs[name] = m.recommend_for_users(eval_users, top_k=50)
catalog = set(train_items); item_pop = train.groupby("movie_id").size().to_dict()
rows = []
for name, recs in all_recs.items():
    r = evaluate_ranking(recs, test_subset, k=10)
    r["model"] = name; r["cov"] = catalog_coverage(recs, catalog, k=10)
    r["nov"] = novelty(recs, item_pop, k=10); rows.append(r)
df = pd.DataFrame(rows).set_index("model")
df = df[["precision@10","recall@10","ndcg@10","hit_rate@10","map@10","cov","nov"]]
display(df.round(4))

## 6. Visualisation des embeddings en 2D (PCA)

Les embeddings appris par les modeles neuroniques capturent souvent des dimensions interpretables.

In [ ]:
from sklearn.decomposition import PCA

emb = twotower.model.item_emb.weight.detach().numpy()
pca = PCA(n_components=2, random_state=0)
proj = pca.fit_transform(emb)

movies_df = ml.movies.set_index("movie_id")
primary_genre = movies_df["genres_list"].apply(lambda g: g[0] if isinstance(g, list) and g else "Unknown")

fig, ax = plt.subplots(figsize=(10, 7))
colors = {"Drama":"#1f77b4","Comedy":"#ff7f0e","Action":"#2ca02c","Thriller":"#d62728",
          "Sci-Fi":"#9467bd","Romance":"#8c564b","Horror":"#e377c2"}
for genre, color in colors.items():
    mask = primary_genre == genre
    if mask.sum() > 0:
        ax.scatter(proj[mask, 0], proj[mask, 1], s=8, alpha=0.5, label=f"{genre} ({mask.sum()})", color=color)
ax.legend(); ax.set_title("TwoTower - items en 2D (PCA)"); plt.show()

## 7. Conclusions Module 4

**Resultats observes (100k sample) :**

| Modele            | RMSE  | MAE   | P@10  | NDCG@10 |
|-------------------|-------|-------|-------|---------|
| TwoTower          | 1.063 | 0.848 | 0.009 | 0.014   |
| NeuralCF          | 0.980 | 0.788 | 0.010 | 0.016   |
| Surprise SVD      | 0.951 | 0.743 | 0.009 | 0.013   |

**Lecons :**

- **NeuralCF > TwoTower** sur RMSE grace a son MLP non-lineaire, mais a plus de parametres et plus lent.
- **Surprise SVD** reste le plus rapide et le plus precis RMSE grace a son implementation en C optimisee.
- Les chiffres de ranking sont tres bas a cause de l echantillon reduit (peu de notes test par user). Sur full data, Surprise SVD donne P@10 ~0.08 (cf. module 2).
- En pratique, sur GPU, NeuralCF peut rivaliser et aller au-dela quand on a beaucoup de features (genres, demographie).

**A venir (Module 5) - LightFM :**

- Modele **hybride** : combine interactions (collaboratif) + features (content-based).
- Resout le **cold start** (utilisateur ou item nouveau).
- Recommandable pour systemes avec metadonnees riches (films, produits, articles).